<a href="https://colab.research.google.com/github/Melikenzli/belediye-rag/blob/main/belediye_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Melikenzli/belediye-rag.git
%cd belediye-rag

Cloning into 'belediye-rag'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 14 (delta 0), reused 14 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 6.75 KiB | 6.75 MiB/s, done.
/content/belediye-rag


In [ ]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.6/79.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
import os
from google.colab import userdata

# Secrets'tan anahtari oku ve ortam degiskenine ata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

# Kontrol: anahtar geldi mi? (tamamini gostermeden)
anahtar = os.environ["GOOGLE_API_KEY"]
print("Anahtar okundu:", anahtar[:6] + "..." + anahtar[-4:])

Anahtar okundu: AQ.Ab8...It-Q


In [ ]:
import google.generativeai as genai
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Embedding destekleyen modeller:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(" -", m.name)


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Embedding destekleyen modeller:
 - models/gemini-embedding-001
 - models/gemini-embedding-2-preview
 - models/gemini-embedding-2


In [ ]:
import pickle
import numpy as np
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings


print("1/4 Dokumanlar yukleniyor...")
yukleyici = DirectoryLoader(
    "data", glob="*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
dokumanlar = yukleyici.load()
print(f"    {len(dokumanlar)} dokuman yuklendi.")


print("2/4 Parcalara ayriliyor...")
ayirici = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
parcalar = ayirici.split_documents(dokumanlar)
print(f"    {len(parcalar)} parca olusturuldu.")


print("3/4 Gemini embedding modeli hazirlaniyor...")
embedding_modeli = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")


print("4/4 Vektorler hesaplanip kaydediliyor...")
metinler = [p.page_content for p in parcalar]
kaynaklar = [p.metadata.get("source", "bilinmiyor") for p in parcalar]
vektorler = np.array(embedding_modeli.embed_documents(metinler), dtype="float32")

with open("vektor_db.pkl", "wb") as f:
    pickle.dump({"vektorler": vektorler, "metinler": metinler, "kaynaklar": kaynaklar}, f)

print(f"\nTamamlandi! {len(metinler)} parca vektorlestirildi.")
print(f"Vektor boyutu: {vektorler.shape}")

/tmp/ipykernel_667/3918464418.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


1/4 Dokumanlar yukleniyor...
    5 dokuman yuklendi.
2/4 Parcalara ayriliyor...
    20 parca olusturuldu.
3/4 Gemini embedding modeli hazirlaniyor...
4/4 Vektorler hesaplanip kaydediliyor...

Tamamlandi! 20 parca vektorlestirildi.
Vektor boyutu: (20, 3072)


In [ ]:
print("Yanit uretebilen (chat) modeller:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(" -", m.name)

Yanit uretebilen (chat) modeller:
 - models/gemini-2.5-flash
 - models/gemini-2.5-pro
 - models/gemini-2.5-flash-preview-tts
 - models/gemini-2.5-pro-preview-tts
 - models/gemma-4-26b-a4b-it
 - models/gemma-4-31b-it
 - models/gemini-flash-latest
 - models/gemini-flash-lite-latest
 - models/gemini-pro-latest
 - models/gemini-2.5-flash-lite
 - models/gemini-2.5-flash-image
 - models/gemini-3-flash-preview
 - models/gemini-3.1-pro-preview
 - models/gemini-3.1-pro-preview-customtools
 - models/gemini-3.1-flash-lite-preview
 - models/gemini-3.1-flash-lite
 - models/gemini-3-pro-image-preview
 - models/gemini-3-pro-image
 - models/nano-banana-pro-preview
 - models/gemini-3.1-flash-image-preview
 - models/gemini-3.1-flash-image
 - models/gemini-3.1-flash-lite-image
 - models/gemini-3.5-flash
 - models/gemini-3.5-flash-lite
 - models/gemini-omni-flash-preview
 - models/gemini-omni-1.1-flash
 - models/gemini-3.5-transcribe
 - models/gemini-3.6-flash
 - models/gemini-3.7-flash
 - models/lyria-3-

In [ ]:
import pickle
import numpy as np
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

# 1) Vektor deposunu yukle
with open("vektor_db.pkl", "rb") as f:
    db = pickle.load(f)
vektorler = db["vektorler"]
metinler = db["metinler"]
kaynaklar = db["kaynaklar"]

# 2) Modeller
embedding_modeli = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
llm = ChatGoogleGenerativeAI(model="models/gemini-flash-latest")

# 3) Ilgili parcalari bul
def ilgili_parcalari_bul(soru, k=3):
    soru_vektoru = np.array(embedding_modeli.embed_query(soru), dtype="float32")
    benzerlikler = vektorler @ soru_vektoru
    en_iyi = np.argsort(benzerlikler)[::-1][:k]
    return [(metinler[i], kaynaklar[i]) for i in en_iyi]

# 4) RAG: bul + Gemini'ye sor
def yanitla(soru):
    parcalar = ilgili_parcalari_bul(soru)
    baglam = "\n\n".join(f"[Kaynak: {k}]\n{m}" for m, k in parcalar)
    istem = f"""Sen bir belediyenin vatandaş hizmetleri asistanısın.
Aşağıdaki BAĞLAM bilgilerini kullanarak vatandaşın SORUSUNU Türkçe yanıtla.
Yalnızca BAĞLAM'daki bilgilere dayan. Bağlamda cevap yoksa, "Bu konuda elimde bilgi yok, lütfen belediyeye başvurun." de.

BAĞLAM:
{baglam}

SORU: {soru}

YANIT:"""
    yanit = llm.invoke(istem)
    icerik = yanit.content
    if isinstance(icerik, list):
        icerik = "".join(p.get("text", "") for p in icerik if isinstance(p, dict))
    return icerik, parcalar

# 5) Test
soru = "Su faturamı nasıl öderim?"
yanit, kullanilan = yanitla(soru)

print(f"SORU: {soru}\n")
print("YANIT:")
print(yanit)
print("\n" + "="*60)
print("KULLANILAN KAYNAKLAR:")
for m, k in kullanilan:
    print(f"  - {k}")

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Quota exceeded for aiplatform.googleapis.com/global_embed_content_requests_per_minute_per_base_model with base model: gemini-embedding. Please submit a quota increase request. https://cloud.google.com/vertex-ai/docs/generative-ai/quotas-genai.', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
!pip install -q sentence-transformers langchain-huggingface

In [ ]:
import pickle
import numpy as np
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# 1) Dokumanlari yukle
print("1/4 Dokumanlar yukleniyor...")
yukleyici = DirectoryLoader(
    "data", glob="*.txt", loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
dokumanlar = yukleyici.load()
print(f"    {len(dokumanlar)} dokuman yuklendi.")


print("2/4 Parcalara ayriliyor...")
ayirici = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
parcalar = ayirici.split_documents(dokumanlar)
print(f"    {len(parcalar)} parca olusturuldu.")


print("3/4 Yerel gomme modeli yukleniyor (ilk seferde iner)...")
embedding_modeli = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    encode_kwargs={"normalize_embeddings": True},
)


print("4/4 Vektorler hesaplanip kaydediliyor...")
metinler = [p.page_content for p in parcalar]
kaynaklar = [p.metadata.get("source", "bilinmiyor") for p in parcalar]
vektorler = np.array(embedding_modeli.embed_documents(metinler), dtype="float32")

with open("vektor_db.pkl", "wb") as f:
    pickle.dump({"vektorler": vektorler, "metinler": metinler, "kaynaklar": kaynaklar}, f)

print(f"\nTamamlandi! {len(metinler)} parca vektorlestirildi.")
print(f"Vektor boyutu: {vektorler.shape}")

1/4 Dokumanlar yukleniyor...
    5 dokuman yuklendi.
2/4 Parcalara ayriliyor...
    20 parca olusturuldu.
3/4 Yerel gomme modeli yukleniyor (ilk seferde iner)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

4/4 Vektorler hesaplanip kaydediliyor...

Tamamlandi! 20 parca vektorlestirildi.
Vektor boyutu: (20, 384)


In [5]:
import pickle
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

# 1) Vektor deposunu yukle
with open("vektor_db.pkl", "rb") as f:
    db = pickle.load(f)
vektorler = db["vektorler"]
metinler = db["metinler"]
kaynaklar = db["kaynaklar"]


embedding_modeli = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    encode_kwargs={"normalize_embeddings": True},
)
llm = ChatGoogleGenerativeAI(model="models/gemini-3.6-flash")

def ilgili_parcalari_bul(soru, k=3):
    soru_vektoru = np.array(embedding_modeli.embed_query(soru), dtype="float32")
    benzerlikler = vektorler @ soru_vektoru
    en_iyi = np.argsort(benzerlikler)[::-1][:k]
    return [(metinler[i], kaynaklar[i]) for i in en_iyi]

def yanitla(soru):
    parcalar = ilgili_parcalari_bul(soru)
    baglam = "\n\n".join(f"[Kaynak: {k}]\n{m}" for m, k in parcalar)
    istem = f"""Sen bir belediyenin vatandaş hizmetleri asistanısın.
Aşağıdaki BAĞLAM bilgilerini kullanarak vatandaşın SORUSUNU Türkçe yanıtla.
Yalnızca BAĞLAM'daki bilgilere dayan. Bağlamda cevap yoksa, "Bu konuda elimde bilgi yok, lütfen belediyeye başvurun." de.

BAĞLAM:
{baglam}

SORU: {soru}

YANIT:"""
    yanit = llm.invoke(istem)
    icerik = yanit.content
    if isinstance(icerik, list):
        icerik = "".join(p.get("text", "") for p in icerik if isinstance(p, dict))
    return icerik, parcalar

# 5) Test
soru = "Su faturamı nasıl öderim?"
yanit, kullanilan = yanitla(soru)

print(f"SORU: {soru}\n")
print("YANIT:")
print(yanit)
print("\n" + "="*60)
print("KULLANILAN KAYNAKLAR:")
for m, k in kullanilan:
    print(f"  - {k}")

ModuleNotFoundError: No module named 'langchain_huggingface'

In [ ]:
sorular = [
    "İmar durumu belgesi nasıl alınır?",
    "Evlenmek için hangi belgeler gerekiyor?",
    "Emlak vergisini ne zaman ödemem gerekiyor?",
    "Eski koltuğumu nasıl attırabilirim?",
    "Bugün hava nasıl olacak?",          # KAPSAM DIŞI - bilmiyorum demeli
]

for soru in sorular:
    yanit, kullanilan = yanitla(soru)
    kaynak_listesi = ", ".join(sorted(set(k for _, k in kullanilan)))
    print(f"SORU: {soru}")
    print(f"YANIT: {yanit}")
    print(f"KAYNAK: {kaynak_listesi}")
    print("=" * 70)

SORU: İmar durumu belgesi nasıl alınır?
YANIT: İmar durumu belgesi almak için tapu fotokopisi ve dilekçe ile İmar Müdürlüğüne başvurmanız gerekmektedir. 

Bu belge; taşınmazın üzerinde ne tür yapı yapılabileceğini, kat sayısını ve yapılaşma koşullarını gösterir. Başvurular genellikle beş iş günü içinde sonuçlandırılmaktadır.
KAYNAK: data/imar_durumu.txt
SORU: Evlenmek için hangi belgeler gerekiyor?
YANIT: Evlenmek için gerekli olan belgeler şunlardır:

* Nüfus cüzdanı
* Altı adet vesikalık fotoğraf
* Evlenme ehliyet belgesi
* Sağlık raporu

Ayrıca yabancı uyruklu kişiler için ek belgeler istenebilmektedir.
KAYNAK: data/evlendirme.txt
SORU: Emlak vergisini ne zaman ödemem gerekiyor?
YANIT: Emlak vergisi yılda iki taksit halinde ödenmektedir:

* **1. Taksit:** Mart, Nisan ve Mayıs aylarında,
* **2. Taksit:** Kasım ayı içinde ödenmektedir. 

Ödemelerinizi zamanında yapmanız durumunda herhangi bir gecikme faizi uygulanmaz.
KAYNAK: data/emlak_vergisi.txt
SORU: Eski koltuğumu nasıl attırabil

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr

def arayuz_yanit(soru):
    if not soru.strip():
        return "Lütfen bir soru yazın.", ""
    yanit, kullanilan = yanitla(soru)
    # Kaynaklari tekilleştir ve listele
    kaynaklar_temiz = sorted(set(k for _, k in kullanilan))
    kaynak_metni = "\n".join(f"• {k}" for k in kaynaklar_temiz)
    return yanit, kaynak_metni

# Arayuzu tasarla
with gr.Blocks(title="Belediye Asistanı") as arayuz:
    gr.Markdown("# 🏛️ Belediye Vatandaş Hizmetleri Asistanı")
    gr.Markdown("Belediye hizmetleriyle ilgili sorularınızı yazın, yanıtlayayım.")

    soru_kutusu = gr.Textbox(
        label="Sorunuz",
        placeholder="Örn: Su faturamı nasıl öderim?",
        lines=2,
    )
    sor_dugmesi = gr.Button("Sor", variant="primary")

    yanit_alani = gr.Textbox(label="Yanıt", lines=6)
    kaynak_alani = gr.Textbox(label="Kaynaklar", lines=3)

    # Ornek sorular
    gr.Examples(
        examples=[
            "Su faturamı nasıl öderim?",
            "İmar durumu belgesi nasıl alınır?",
            "Evlenmek için hangi belgeler gerekiyor?",
            "Eski koltuğumu nasıl attırabilirim?",
        ],
        inputs=soru_kutusu,
    )

    # Dugmeye basilinca calis
    sor_dugmesi.click(fn=arayuz_yanit, inputs=soru_kutusu, outputs=[yanit_alani, kaynak_alani])

# Arayuzu baslat
arayuz.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24a33c9d8d920b8e65.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [1]:
# Test kumesi: (soru, beklenen kaynak dosya)
test_kumesi = [
    ("Su faturamı nasıl öderim?", "su_islemleri.txt"),
    ("Su aboneliği için ne gerekiyor?", "su_islemleri.txt"),
    ("Emlak vergisini ne zaman ödemeliyim?", "emlak_vergisi.txt"),
    ("Çevre temizlik vergisi nedir?", "emlak_vergisi.txt"),
    ("İmar durumu belgesi nasıl alınır?", "imar_durumu.txt"),
    ("Yapı ruhsatı için hangi belgeler lazım?", "imar_durumu.txt"),
    ("Evlenmek için gerekli belgeler neler?", "evlendirme.txt"),
    ("Nikah töreni nerede yapılır?", "evlendirme.txt"),
    ("Çöp toplama günlerini nasıl öğrenirim?", "cop_toplama.txt"),
    ("Eski koltuğumu nasıl attırırım?", "cop_toplama.txt"),
]

print(f"Test kümesinde {len(test_kumesi)} soru var.")
print("Her sorunun beklenen kaynağı önceden tanımlandı (ground truth).")

Test kümesinde 10 soru var.
Her sorunun beklenen kaynağı önceden tanımlandı (ground truth).


In [2]:
import time

sonuclar = []          # her soru icin detay saklanacak
retrieval_dogru = 0    # dogru dosya bulunan soru sayisi
sureler = []           # her sorunun yanit suresi

print("Değerlendirme başlıyor...\n")

for i, (soru, beklenen_kaynak) in enumerate(test_kumesi, 1):
    baslangic = time.time()
    yanit, kullanilan = yanitla(soru)
    gecen_sure = time.time() - baslangic
    sureler.append(gecen_sure)

    # Retrieval kontrolu: getirilen kaynaklardan biri beklenen dosya mi?
    bulunan_kaynaklar = [k for _, k in kullanilan]
    retrieval_isabet = any(beklenen_kaynak in k for k in bulunan_kaynaklar)
    if retrieval_isabet:
        retrieval_dogru += 1

    sonuclar.append({
        "soru": soru,
        "beklenen": beklenen_kaynak,
        "retrieval_isabet": retrieval_isabet,
        "yanit": yanit,
        "sure": gecen_sure,
    })

    durum = "✓" if retrieval_isabet else "✗"
    print(f"{i:2}. {durum}  ({gecen_sure:.1f}sn)  {soru}")

    time.sleep(2)  # kotayi zorlamamak icin kisa bekleme

# --- RETRIEVAL DOGRULUGU ---
retrieval_oran = retrieval_dogru / len(test_kumesi) * 100
print("\n" + "="*55)
print("RETRIEVAL DOĞRULUĞU (doğru dosya bulundu mu?)")
print(f"  {retrieval_dogru}/{len(test_kumesi)} soru = %{retrieval_oran:.0f}")

# --- GECIKME ---
print("\nGECİKME (yanıt süresi)")
print(f"  Ortalama: {sum(sureler)/len(sureler):.2f} sn")
print(f"  En hızlı: {min(sureler):.2f} sn")
print(f"  En yavaş: {max(sureler):.2f} sn")

Değerlendirme başlıyor...



NameError: name 'yanitla' is not defined

In [6]:
import pickle
import numpy as np
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

# 1) Vektor deposunu yukle
with open("vektor_db.pkl", "rb") as f:
    db = pickle.load(f)
vektorler = db["vektorler"]
metinler = db["metinler"]
kaynaklar = db["kaynaklar"]


embedding_modeli = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    encode_kwargs={"normalize_embeddings": True},
)
llm = ChatGoogleGenerativeAI(model="models/gemini-3.6-flash")

def ilgili_parcalari_bul(soru, k=3):
    soru_vektoru = np.array(embedding_modeli.embed_query(soru), dtype="float32")
    benzerlikler = vektorler @ soru_vektoru
    en_iyi = np.argsort(benzerlikler)[::-1][:k]
    return [(metinler[i], kaynaklar[i]) for i in en_iyi]

def yanitla(soru):
    parcalar = ilgili_parcalari_bul(soru)
    baglam = "\n\n".join(f"[Kaynak: {k}]\n{m}" for m, k in parcalar)
    istem = f"""Sen bir belediyenin vatandaş hizmetleri asistanısın.
Aşağıdaki BAĞLAM bilgilerini kullanarak vatandaşın SORUSUNU Türkçe yanıtla.
Yalnızca BAĞLAM'daki bilgilere dayan. Bağlamda cevap yoksa, "Bu konuda elimde bilgi yok, lütfen belediyeye başvurun." de.

BAĞLAM:
{baglam}

SORU: {soru}

YANIT:"""
    yanit = llm.invoke(istem)
    icerik = yanit.content
    if isinstance(icerik, list):
        icerik = "".join(p.get("text", "") for p in icerik if isinstance(p, dict))
    return icerik, parcalar

# 5) Test
soru = "Su faturamı nasıl öderim?"
yanit, kullanilan = yanitla(soru)

print(f"SORU: {soru}\n")
print("YANIT:")
print(yanit)
print("\n" + "="*60)
print("KULLANILAN KAYNAKLAR:")
for m, k in kullanilan:
    print(f"  - {k}")

ModuleNotFoundError: No module named 'langchain_huggingface'

In [7]:
!git clone https://github.com/Melikenzli/belediye-rag.git
%cd belediye-rag

Cloning into 'belediye-rag'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 17 (delta 1), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 27.55 KiB | 4.59 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/belediye-rag


In [8]:
!pip install -q langchain langchain-community langchain-google-genai langchain-text-splitters sentence-transformers langchain-huggingface gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.
google-colab 1

In [9]:
import os
from google.colab import userdata
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("Anahtar yüklendi.")

Anahtar yüklendi.


In [11]:
import pickle, numpy as np
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

yukleyici = DirectoryLoader("data", glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
parcalar = RecursiveCharacterTextSplitter(
    chunk_size=400, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""]
).split_documents(yukleyici.load())

embedding_modeli = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    encode_kwargs={"normalize_embeddings": True},
)
metinler = [p.page_content for p in parcalar]
kaynaklar = [p.metadata.get("source", "bilinmiyor") for p in parcalar]
vektorler = np.array(embedding_modeli.embed_documents(metinler), dtype="float32")
with open("vektor_db.pkl", "wb") as f:
    pickle.dump({"vektorler": vektorler, "metinler": metinler, "kaynaklar": kaynaklar}, f)
print(f"Vektör deposu hazır: {vektorler.shape}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vektör deposu hazır: (20, 384)


In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="models/gemini-3.6-flash")

def ilgili_parcalari_bul(soru, k=3):
    sv = np.array(embedding_modeli.embed_query(soru), dtype="float32")
    benzerlikler = vektorler @ sv
    en_iyi = np.argsort(benzerlikler)[::-1][:k]
    return [(metinler[i], kaynaklar[i]) for i in en_iyi]

def yanitla(soru):
    parcalar = ilgili_parcalari_bul(soru)
    baglam = "\n\n".join(f"[Kaynak: {k}]\n{m}" for m, k in parcalar)
    istem = f"""Sen bir belediyenin vatandaş hizmetleri asistanısın.
Aşağıdaki BAĞLAM bilgilerini kullanarak vatandaşın SORUSUNU Türkçe yanıtla.
Yalnızca BAĞLAM'daki bilgilere dayan. Bağlamda cevap yoksa, "Bu konuda elimde bilgi yok, lütfen belediyeye başvurun." de.

BAĞLAM:
{baglam}

SORU: {soru}

YANIT:"""
    yanit = llm.invoke(istem)
    icerik = yanit.content
    if isinstance(icerik, list):
        icerik = "".join(p.get("text", "") for p in icerik if isinstance(p, dict))
    return icerik, parcalar

print("yanitla fonksiyonu hazır.")

yanitla fonksiyonu hazır.


In [13]:
test_kumesi = [
    ("Su faturamı nasıl öderim?", "su_islemleri.txt"),
    ("Su aboneliği için ne gerekiyor?", "su_islemleri.txt"),
    ("Emlak vergisini ne zaman ödemeliyim?", "emlak_vergisi.txt"),
    ("Çevre temizlik vergisi nedir?", "emlak_vergisi.txt"),
    ("İmar durumu belgesi nasıl alınır?", "imar_durumu.txt"),
    ("Yapı ruhsatı için hangi belgeler lazım?", "imar_durumu.txt"),
    ("Evlenmek için gerekli belgeler neler?", "evlendirme.txt"),
    ("Nikah töreni nerede yapılır?", "evlendirme.txt"),
    ("Çöp toplama günlerini nasıl öğrenirim?", "cop_toplama.txt"),
    ("Eski koltuğumu nasıl attırırım?", "cop_toplama.txt"),
]
print(f"Test kümesinde {len(test_kumesi)} soru var.")

Test kümesinde 10 soru var.


In [14]:
import time

sonuclar = []
retrieval_dogru = 0
sureler = []

print("Değerlendirme başlıyor...\n")

for i, (soru, beklenen_kaynak) in enumerate(test_kumesi, 1):
    baslangic = time.time()
    yanit, kullanilan = yanitla(soru)
    gecen_sure = time.time() - baslangic
    sureler.append(gecen_sure)

    bulunan_kaynaklar = [k for _, k in kullanilan]
    retrieval_isabet = any(beklenen_kaynak in k for k in bulunan_kaynaklar)
    if retrieval_isabet:
        retrieval_dogru += 1

    sonuclar.append({
        "soru": soru, "beklenen": beklenen_kaynak,
        "retrieval_isabet": retrieval_isabet, "yanit": yanit, "sure": gecen_sure,
    })

    durum = "✓" if retrieval_isabet else "✗"
    print(f"{i:2}. {durum}  ({gecen_sure:.1f}sn)  {soru}")
    time.sleep(2)

retrieval_oran = retrieval_dogru / len(test_kumesi) * 100
print("\n" + "="*55)
print("RETRIEVAL DOĞRULUĞU (doğru dosya bulundu mu?)")
print(f"  {retrieval_dogru}/{len(test_kumesi)} soru = %{retrieval_oran:.0f}")
print("\nGECİKME (yanıt süresi)")
print(f"  Ortalama: {sum(sureler)/len(sureler):.2f} sn")
print(f"  En hızlı: {min(sureler):.2f} sn")
print(f"  En yavaş: {max(sureler):.2f} sn")

Değerlendirme başlıyor...



 1. ✓  (25.4sn)  Su faturamı nasıl öderim?
 2. ✓  (3.7sn)  Su aboneliği için ne gerekiyor?
 3. ✓  (3.5sn)  Emlak vergisini ne zaman ödemeliyim?
 4. ✓  (4.7sn)  Çevre temizlik vergisi nedir?
 5. ✓  (5.1sn)  İmar durumu belgesi nasıl alınır?
 6. ✓  (4.6sn)  Yapı ruhsatı için hangi belgeler lazım?
 7. ✓  (5.4sn)  Evlenmek için gerekli belgeler neler?
 8. ✓  (3.1sn)  Nikah töreni nerede yapılır?
 9. ✓  (11.1sn)  Çöp toplama günlerini nasıl öğrenirim?
10. ✓  (4.7sn)  Eski koltuğumu nasıl attırırım?

RETRIEVAL DOĞRULUĞU (doğru dosya bulundu mu?)
  10/10 soru = %100

GECİKME (yanıt süresi)
  Ortalama: 7.14 sn
  En hızlı: 3.14 sn
  En yavaş: 25.41 sn
